In [ ]:
import pandas as pd
import utils
import os

%load_ext autoreload
%autoreload 2

In [ ]:
data_folder = 'data/'
raw_train_df = pd.read_csv(data_folder + 'train.csv')
raw_test_df = pd.read_csv(data_folder + 'test.csv')
train_demo_df = pd.read_csv(data_folder + 'train_demographics.csv')
test_demo_df = pd.read_csv(data_folder + 'test_demographics.csv')

In [ ]:
feat_columns = ['sequence_type', 'sequence_id', 'sequence_counter', 'subject', 'orientation', 'behavior', 'phase', 'gesture']
acc_columns = ['acc_x', 'acc_y', 'acc_z']
rot_columns = ['rot_x', 'rot_y', 'rot_z']
temp_columns = ['thm_1', 'thm_2', 'thm_3', 'thm_4', 'thm_5']
tof_columns = raw_train_df.columns[raw_train_df.columns.str.startswith('tof')]
non_device_cols = ['sequence_type', 'sequence_id', 'sequence_counter', 'subject', 'orientation', 'behavior', 'phase', 'gesture', 'adult_child', 'age', 'sex', 'handedness', 'height_cm', 'shoulder_to_wrist_cm', 'elbow_to_wrist_cm']


# Explore

In [ ]:
train_df = raw_train_df.set_index('row_id')
train_df.head(2)

In [ ]:
train_demo_df.head(1)

In [ ]:
train_df['sequence_type'].value_counts()

In [ ]:
train_df['sequence_id'].nunique(), train_df['sequence_counter'].nunique(), train_df['subject'].nunique(), train_df['orientation'].nunique(), train_df['behavior'].nunique()

In [ ]:
train_df['orientation'].value_counts()

In [ ]:
train_df['behavior'].value_counts()

In [ ]:
train_df[feat_columns]

In [ ]:
example_sequence = 'SEQ_000007'
example_seq_df = train_df[train_df['sequence_id'] == example_sequence].merge(train_demo_df, how='left', on='subject')
example_seq_df.groupby(['gesture']).agg(**{
    'Gesture': ('gesture', 'unique'),
    'sequence':('sequence_id', 'first'),
    'subject':('subject', 'unique'),
    'orientation':('orientation', 'unique'),
    'sequence':('sequence_id', 'unique'),
    'sequence_num':('sequence_id', 'count'),
    'behavior_num':('behavior', 'count'),
    'phase':('phase', 'unique'),
    'adult_child':('adult_child', 'first'),
    'age':('age', 'first'),
    'sex':('sex', 'first'),
    'handedness':('handedness', 'first'),
    'height_cm':('height_cm', 'first'),
    'shoulder_to_wrist_cm':('shoulder_to_wrist_cm', 'first'),
    'elbow_to_wrist_cm':('elbow_to_wrist_cm', 'first')
})

In [ ]:
(train_demo_df.groupby(['subject']).agg(**{
    'adult_child':('adult_child','nunique'),
    'age':('age', 'nunique'),
    'sex':('sex', 'nunique'),
    'handedness':('handedness', 'nunique'),
    'height_cm':('height_cm', 'nunique'),
    'shoulder_to_wrist_cm':('shoulder_to_wrist_cm', 'nunique'),
    'elbow_to_wrist_cm':('elbow_to_wrist_cm', 'nunique')
}) > 1).sum().sum()

In [ ]:
example_seq_df.head(1)

In [ ]:
example_seq_df.loc[example_seq_df['sequence_id'] == example_sequence, non_device_cols].head()

In [ ]:
example_seq_df.shape

In [ ]:
train_df.groupby('gesture').agg(**{
    'Recordings': ('sequence_id', 'nunique'),
})

In [ ]:
train_df[train_df['sequence_id'] == 'SEQ_065526'].phase.unique()

In [ ]:
phase_eg = train_df.groupby('sequence_id').agg({'phase': 'nunique','behavior': 'nunique', 'orientation': 'nunique'})
phase_eg[phase_eg['phase'] > 2]

In [ ]:
phase_eg[phase_eg['behavior'] > 3]

In [ ]:
phase_eg[phase_eg['orientation'] > 1]

In [ ]:
train_df[train_df['phase'] == 'Pause']

In [ ]:
train_df['subject'].nunique()

In [ ]:
# Count rows per phase for different sequences
train_df.groupby(['sequence_id', 'behavior']).size()

In [ ]:
example_seq_df[['acc_x', 'acc_y', 'acc_z']].plot()

In [ ]:
example_seq_df[['acc_x', 'acc_y', 'acc_z']].head()

In [ ]:
sampling_rate = 1000
example_fft_df = example_seq_df[['acc_x', 'acc_y', 'acc_z']].apply(utils.convert_frame_to_fft, axis=0, args=(sampling_rate,))
# example_fft_df.loc[0:5] = 0
example_fft_df.head()

In [ ]:
features = (
    example_fft_df
    .apply(lambda col: utils.extract_features(col, sampling_rate=sampling_rate))
    .T
    .add_prefix("")
)

single_row = features.stack()
single_row.index = [f"{axis}_{feat}" for axis, feat in single_row.index]
single_row = single_row.to_frame().T